In [197]:
import pandas as pd 
import numpy as np
import pyarrow as pa 
import pyarrow.parquet as pq
import duckdb
import faker
import sys
import os
import time

In [198]:
fake = faker.Faker()
cities = [fake.city() for i in range(10)]

In [199]:
n = 500_000
start_date = np.datetime64('2023-01-01')
end_date = np.datetime64('2026-03-01')

signup_dates = start_date + np.random.randint(0, (end_date - start_date).astype(int), size=n).astype('timedelta64[D]')

dataset = {
    'user_id': np.arange(n),
    'city': np.random.choice(cities, size=n),
    'score': np.random.uniform(0, 100, n),
    'active': np.random.randint(0, 2, n),
    'signup_date': signup_dates,
    'age': np.random.randint(18, 80, n),
    'sessions': np.random.randint(0, 500, n),
    'revenue': np.random.uniform(0, 1000, n)
}

dataset

{'user_id': array([     0,      1,      2, ..., 499997, 499998, 499999],
       shape=(500000,)),
 'city': array(['East Timothy', 'West Erinborough', 'Caseyport', ...,
        'Carolynland', 'New Brandon', 'West Erinborough'],
       shape=(500000,), dtype='<U19'),
 'score': array([53.36670363, 51.81765287, 82.46591259, ..., 37.25973819,
        74.78344089, 36.90206361], shape=(500000,)),
 'active': array([0, 0, 1, ..., 1, 1, 1], shape=(500000,)),
 'signup_date': array(['2025-05-19', '2023-02-27', '2023-04-27', ..., '2025-03-26',
        '2025-08-30', '2024-02-27'], shape=(500000,), dtype='datetime64[D]'),
 'age': array([22, 42, 28, ..., 26, 72, 35], shape=(500000,)),
 'sessions': array([ 16, 262, 459, ..., 269,  34, 295], shape=(500000,)),
 'revenue': array([230.90399836, 229.71231207, 310.74848666, ..., 309.57606627,
         96.57017746, 703.77551326], shape=(500000,))}

In [200]:
df = pd.DataFrame(dataset)

In [201]:
df.to_parquet('dataset.parquet', index=False)

In [202]:
pf = pq.ParquetFile("dataset.parquet") 
print(f"Row groups: {pf.num_row_groups}")
print(f"Rows: {pf.metadata.num_rows}")
print(f"Columns: {pf.metadata.num_columns}")
print(f"Column names: {pf.schema.names}")
print(f"Column types: {[str(t) for t in pf.schema_arrow.types]}")


Row groups: 1
Rows: 500000
Columns: 8
Column names: ['user_id', 'city', 'score', 'active', 'signup_date', 'age', 'sessions', 'revenue']
Column types: ['int64', 'large_string', 'double', 'int64', 'timestamp[ms]', 'int64', 'int64', 'double']


In [203]:
pf.metadata.row_group(0).column(0)

  file_offset: 0
  file_path: 
  physical_type: INT64
  num_values: 500000
  path_in_schema: user_id
  is_stats_set: True
  statistics:
      has_min_max: True
      min: 0
      max: 499999
      null_count: 0
      distinct_count: None
      num_values: 500000
      physical_type: INT64
      logical_type: None
      converted_type (legacy): NONE
  geo_statistics:
    None
  compression: SNAPPY
  encodings: ('PLAIN', 'RLE', 'RLE_DICTIONARY')
  has_dictionary_page: True
  dictionary_page_offset: 4
  data_page_offset: 525342
  total_compressed_size: 2273833
  total_uncompressed_size: 4272637

In [204]:
pf.metadata.row_group(0).column(0).path_in_schema

'user_id'

In [205]:
rg = pf.metadata.row_group(0)

for i in range(rg.num_columns):
    col = rg.column(i)
    stats = col.statistics

    print(f"Column: {col.path_in_schema}")
    print(f"  Physical type:    {col.physical_type}")
    print(f"  Compressed size:  {col.total_compressed_size} bytes")

    if stats and stats.has_min_max:
        print(f"  Min:              {stats.min}")
        print(f"  Max:              {stats.max}")
    else:
        print(f"  Min/Max:          unavailable")

    if stats:
        print(f"  Null count:       {stats.null_count}")
    else:
        print(f"  Null count:       unavailable")

    print()

Column: user_id
  Physical type:    INT64
  Compressed size:  2273833 bytes
  Min:              0
  Max:              499999
  Null count:       0

Column: city
  Physical type:    BYTE_ARRAY
  Compressed size:  253038 bytes
  Min:              Carolynland
  Max:              West Erinborough
  Null count:       0

Column: score
  Physical type:    DOUBLE
  Compressed size:  4272959 bytes
  Min:              2.9664612433144555e-06
  Max:              99.99996800540679
  Null count:       0

Column: active
  Physical type:    INT64
  Compressed size:  66216 bytes
  Min:              0
  Max:              1
  Null count:       0

Column: signup_date
  Physical type:    INT64
  Compressed size:  697570 bytes
  Min:              2023-01-01 00:00:00
  Max:              2026-02-28 00:00:00
  Null count:       0

Column: age
  Physical type:    INT64
  Compressed size:  378356 bytes
  Min:              18
  Max:              79
  Null count:       0

Column: sessions
  Physical type:    INT64

In [206]:
df.to_csv('dataset.csv', index=False)

In [207]:
print(f"Size of Parquet: {os.path.getsize('dataset.parquet')} bytes")
print(f"Size of CSV:     {os.path.getsize('dataset.csv')} bytes")

Size of Parquet: 12786963 bytes
Size of CSV:     38650444 bytes


Parquet allows to have access to predefined columns only, and also have information even before uploading the files. Parquet is columnar whilst csv is row based data format.
Parquet is optimized for analytics where you often read only a subset of columns and benefit from metadata-driven optimizations, while CSV is simpler and human-readable but less efficient for big data operations.

In [208]:
%time pq.read_table('dataset.parquet')

CPU times: user 61.9 ms, sys: 52.1 ms, total: 114 ms
Wall time: 47.2 ms


pyarrow.Table
user_id: int64
city: large_string
score: double
active: int64
signup_date: timestamp[ms]
age: int64
sessions: int64
revenue: double
----
user_id: [[0,1,2,3,4,...,131067,131068,131069,131070,131071],[131072,131073,131074,131075,131076,...,262139,262140,262141,262142,262143],[262144,262145,262146,262147,262148,...,393211,393212,393213,393214,393215],[393216,393217,393218,393219,393220,...,499995,499996,499997,499998,499999]]
city: [["East Timothy","West Erinborough","Caseyport","East Timothy","Smithside",...,"West Erinborough","South Michelletown","Port Jameston","Port Jameston","South Michelletown"],["Port Curtismouth","Carolynland","South Michelletown","New Brandon","Port Curtismouth",...,"Port Stephanieshire","West Erinborough","Port Jameston","West Erinborough","Port Curtismouth"],["Carolynland","Port Stephanieshire","East Timothy","Port Curtismouth","Port Stephanieshire",...,"West Erinborough","East Timothy","Carolynland","New Brandon","South Michelletown"],["West Erin

In [209]:
%time pq.read_table('dataset.parquet', columns=['user_id', 'score'])

CPU times: user 16.4 ms, sys: 8.98 ms, total: 25.4 ms
Wall time: 16.9 ms


pyarrow.Table
user_id: int64
score: double
----
user_id: [[0,1,2,3,4,...,131067,131068,131069,131070,131071],[131072,131073,131074,131075,131076,...,262139,262140,262141,262142,262143],[262144,262145,262146,262147,262148,...,393211,393212,393213,393214,393215],[393216,393217,393218,393219,393220,...,499995,499996,499997,499998,499999]]
score: [[53.36670363491418,51.81765287440082,82.4659125886082,36.00672767906161,67.91224833927586,...,1.6756643459519327,43.54678807399359,95.45844422698816,32.34852053988453,50.82749513773906],[59.07627442164471,53.9353808045821,11.319405324229537,85.53685043471219,45.85335590916271,...,25.2185879404625,48.19499068663825,82.97511130259633,5.943828236615523,69.1968778542707],[53.14338822162047,95.1068416341806,5.788208448023346,28.42367642671284,51.93300727794203,...,82.3347725255787,9.134154959297236,55.62944327157984,77.4987466152159,29.082547199223775],[21.12260037463053,44.12300322434077,26.643899097099055,34.303058694534336,58.69105601832282,...,9.3

In [210]:
%time pd.read_csv('dataset.csv')[['user_id', 'score']]

CPU times: user 391 ms, sys: 18.9 ms, total: 410 ms
Wall time: 409 ms


,user_id,score
0,0,53.366704
1,1,51.817653
2,2,82.465913
3,3,36.006728
4,4,67.912248
...,...,...
499995,499995,9.393500
499996,499996,53.792138
499997,499997,37.259738
499998,499998,74.783441


Parquet stores each column in its own chunk with byte offsets recorded in a footer, so when you only need two columns, the reader jumps straight to those and ignores everything else on disk. CSV has none of that structure, it has to scan every single byte just to parse where fields begin and end, even if you're going to throw most of them away immediately after. The wider your table gets, the more painful that difference becomes.

In [211]:
a = time.time()
pq_filtered = pq.read_table("dataset.parquet", filters=[("age", ">", 50)])
pq_time = time.time() - a


In [212]:
a = time.time()
parquet_to_pd = pd.read_parquet('dataset.parquet')
pd_filtered = parquet_to_pd[parquet_to_pd.age > 50]
pd_time = time.time() - a

In [213]:
print(f"Predicate pushdown: {len(pq_filtered)} rows, {pq_time:.4f} sec")
print(f"Post-load filter: {len(pd_filtered)} rows, {pd_time:.4f} sec")

Predicate pushdown: 233719 rows, 0.0395 sec
Post-load filter: 233719 rows, 0.0341 sec


Predicate pushdown is a performance optimization available in columnar storage formats like Parquet.  
When reading a Parquet file, instead of loading the entire dataset into memory, you can provide a filter condition. Parquet uses the metadata stored in the file (such as min/max values for each column in each row group) to skip entire blocks of data that cannot satisfy the filter, reducing I/O and memory usage.
This approach is faster than loading the full dataset into pandas and applying the filter afterward because:
1. Only relevant columns and row groups are read from disk.
2. Unnecessary data is never loaded into memory.
3. Less computation is required since filtering happens during reading, not afterward.

In contrast, if you read the full Parquet file and then filter in pandas, the program must load all rows and columns, which is slower and uses more memory.  

In [214]:
a = time.time()
result = duckdb.sql("SELECT * FROM 'dataset.parquet' WHERE age > 50").df()
duck_time = time.time() - a

In [215]:
print(f"Predicate pushdown: {len(pq_filtered)} rows, {pq_time:.4f} sec")
print(f"Post-load filter: {len(pd_filtered)} rows, {pd_time:.4f} sec")
print(f"duckdb filter: {len(result)} rows, {duck_time:.4f} sec")

Predicate pushdown: 233719 rows, 0.0395 sec
Post-load filter: 233719 rows, 0.0341 sec
duckdb filter: 233719 rows, 0.0708 sec


In [216]:
df = pd.read_parquet('dataset.parquet')
df

,user_id,city,score,active,signup_date,age,sessions,revenue
0,0,East Timothy,53.366704,0,2025-05-19,22,16,230.903998
1,1,West Erinborough,51.817653,0,2023-02-27,42,262,229.712312
2,2,Caseyport,82.465913,1,2023-04-27,28,459,310.748487
3,3,East Timothy,36.006728,1,2023-12-20,20,88,574.583869
4,4,Smithside,67.912248,1,2025-07-18,39,75,121.882296
...,...,...,...,...,...,...,...,...
499995,499995,Smithside,9.393500,0,2026-02-07,44,36,3.350609
499996,499996,New Brandon,53.792138,1,2024-12-27,67,388,904.120410
499997,499997,Carolynland,37.259738,1,2025-03-26,26,269,309.576066
499998,499998,New Brandon,74.783441,1,2025-08-30,72,34,96.570177


In [217]:
a = time.time()
df.groupby('city').size().reset_index(name='total_rows')
time.time() - a

0.01996612548828125

In [218]:
a = time.time()
query = '''
SELECT city, COUNT(*) AS total_rows
FROM "dataset.parquet"
GROUP BY city
ORDER BY city
'''

print(duckdb.sql(query).df())
print(time.time() - a)

                  city  total_rows
0          Carolynland       49768
1            Caseyport       50551
2         East Timothy       49549
3          New Brandon       50278
4     Port Curtismouth       49643
5        Port Jameston       49918
6  Port Stephanieshire       50429
7            Smithside       49975
8   South Michelletown       49868
9     West Erinborough       50021
0.013818740844726562


In [219]:
df.sample()

,user_id,city,score,active,signup_date,age,sessions,revenue
492600,492600,Smithside,1.936735,0,2026-02-03,52,99,214.275307


In [220]:
a = time.time()
print(df.groupby('city')['score'].mean().sort_values(ascending=False))
time.time() - a

city
South Michelletown     50.245925
East Timothy           50.071057
Port Stephanieshire    50.041323
Carolynland            50.037239
Port Jameston          50.022841
Smithside              50.000999
West Erinborough       49.991673
Port Curtismouth       49.972787
Caseyport              49.919847
New Brandon            49.675182
Name: score, dtype: float64


0.023730754852294922

In [221]:

a = time.time()
query = '''
SELECT city, AVG(score) AS avg_score
FROM "dataset.parquet"
GROUP BY city
ORDER BY avg_score DESC
'''

print(duckdb.sql(query).df())
print(time.time() - a)

                  city  avg_score
0   South Michelletown  50.245925
1         East Timothy  50.071057
2  Port Stephanieshire  50.041323
3          Carolynland  50.037239
4        Port Jameston  50.022841
5            Smithside  50.000999
6     West Erinborough  49.991673
7     Port Curtismouth  49.972787
8            Caseyport  49.919847
9          New Brandon  49.675182
0.017299175262451172


In [222]:
a = time.time()
result = (
    df[df['active'] == 1].groupby('city').agg(
          total_active_users=('user_id', 'count'),
          percent_above_75=('score', lambda x: (x > 75).mean() * 100)
      )
      .reset_index()
      .sort_values(by='percent_above_75', ascending=False)
)

print(result)
print(time.time() - a)

                  city  total_active_users  percent_above_75
8   South Michelletown               24863         25.801392
2         East Timothy               24956         25.460811
4     Port Curtismouth               24654         25.338687
1            Caseyport               25151         25.064610
6  Port Stephanieshire               25426         24.958704
9     West Erinborough               24981         24.854890
7            Smithside               25066         24.842416
5        Port Jameston               25016         24.788136
3          New Brandon               25086         24.778761
0          Carolynland               25029         24.723321
0.048140525817871094


In [223]:
a = time.time()
query = '''SELECT 
    city,
    COUNT(*) AS total_active_users,
    100.0 * SUM(CASE WHEN score > 75 THEN 1 ELSE 0 END) / COUNT(*) AS percent_above_75
FROM "dataset.parquet"
WHERE active = 1
GROUP BY city
ORDER BY percent_above_75 DESC;
'''
print(duckdb.sql(query).df())
print(time.time() - a)


                  city  total_active_users  percent_above_75
0   South Michelletown               24863         25.801392
1         East Timothy               24956         25.460811
2     Port Curtismouth               24654         25.338687
3            Caseyport               25151         25.064610
4  Port Stephanieshire               25426         24.958704
5     West Erinborough               24981         24.854890
6            Smithside               25066         24.842416
7        Port Jameston               25016         24.788136
8          New Brandon               25086         24.778761
9          Carolynland               25029         24.723321
0.031228303909301758


In [224]:
a = time.time()
df['score_rank'] = df.groupby('city')['score'].rank(method='min', ascending=False)
top_users_with_rank = df[df['score_rank'] <= 10].sort_values(['city', 'score_rank'])

print(top_users_with_rank)
print("Elapsed time:", time.time() - a)

        user_id              city      score  active signup_date  age  \
212698   212698       Carolynland  99.995997       0  2024-01-02   69   
8             8       Carolynland  99.992859       0  2023-04-11   32   
284483   284483       Carolynland  99.990427       0  2026-02-08   72   
371960   371960       Carolynland  99.987605       1  2023-01-04   60   
422070   422070       Carolynland  99.987288       1  2025-01-20   27   
...         ...               ...        ...     ...         ...  ...   
153473   153473  West Erinborough  99.974966       0  2025-05-21   45   
227750   227750  West Erinborough  99.973588       1  2023-01-03   19   
482996   482996  West Erinborough  99.973127       1  2023-12-13   54   
88046     88046  West Erinborough  99.968564       1  2024-03-10   57   
329546   329546  West Erinborough  99.968457       1  2025-10-07   22   

        sessions     revenue  score_rank  
212698       286  662.488737         1.0  
8            243  770.897419         

In [225]:
a = time.time()
query = '''SELECT *
FROM (
    SELECT
        user_id,
        city,
        score,
        active,
        signup_date,
        age,
        sessions,
        revenue,
        RANK() OVER (PARTITION BY city ORDER BY score DESC) AS score_rank
    FROM "dataset.parquet"
) ranked
WHERE score_rank <= 10
ORDER BY city, score_rank;
'''
print(duckdb.sql(query).df())
print(time.time() - a)


    user_id              city      score  active signup_date  age  sessions  \
0    212698       Carolynland  99.995997       0  2024-01-02   69       286   
1         8       Carolynland  99.992859       0  2023-04-11   32       243   
2    284483       Carolynland  99.990427       0  2026-02-08   72       170   
3    371960       Carolynland  99.987605       1  2023-01-04   60       194   
4    422070       Carolynland  99.987288       1  2025-01-20   27       343   
..      ...               ...        ...     ...         ...  ...       ...   
95   153473  West Erinborough  99.974966       0  2025-05-21   45       376   
96   227750  West Erinborough  99.973588       1  2023-01-03   19       294   
97   482996  West Erinborough  99.973127       1  2023-12-13   54       449   
98    88046  West Erinborough  99.968564       1  2024-03-10   57       341   
99   329546  West Erinborough  99.968457       1  2025-10-07   22       221   

       revenue  score_rank  
0   662.488737        

In [226]:
a = time.time()
df_sorted = df.sort_values(['city', 'user_id']).copy()
df_sorted['running_total_score'] = df_sorted.groupby('city')['score'].cumsum()

print(df_sorted[['user_id', 'city', 'score', 'running_total_score']])
print(time.time() - a)

        user_id              city      score  running_total_score
6             6       Carolynland   1.789002         1.789002e+00
8             8       Carolynland  99.992859         1.017819e+02
14           14       Carolynland  92.257987         1.940398e+02
24           24       Carolynland  75.895653         2.699355e+02
34           34       Carolynland  66.485547         3.364210e+02
...         ...               ...        ...                  ...
499957   499957  West Erinborough  60.346021         2.500505e+06
499970   499970  West Erinborough   6.943974         2.500512e+06
499972   499972  West Erinborough   4.341670         2.500517e+06
499976   499976  West Erinborough  79.967384         2.500597e+06
499999   499999  West Erinborough  36.902064         2.500633e+06

[500000 rows x 4 columns]
0.12968945503234863


In [227]:
a = time.time()
query = '''SELECT
    user_id,
    city,
    score,
    SUM(score) OVER (PARTITION BY city ORDER BY user_id) AS running_total_score
FROM "dataset.parquet"
ORDER BY city, user_id;
'''
print(duckdb.sql(query).df())
print(time.time() - a)

        user_id              city      score  running_total_score
0             6       Carolynland   1.789002         1.789002e+00
1             8       Carolynland  99.992859         1.017819e+02
2            14       Carolynland  92.257987         1.940398e+02
3            24       Carolynland  75.895653         2.699355e+02
4            34       Carolynland  66.485547         3.364210e+02
...         ...               ...        ...                  ...
499995   499957  West Erinborough  60.346021         2.500505e+06
499996   499970  West Erinborough   6.943974         2.500512e+06
499997   499972  West Erinborough   4.341670         2.500517e+06
499998   499976  West Erinborough  79.967384         2.500597e+06
499999   499999  West Erinborough  36.902064         2.500633e+06

[500000 rows x 4 columns]
0.20175981521606445


SQL in DuckDB was generally easier to write because window functions and aggregations can be expressed in a single statement, while pandas often required sorting, grouping, and lambda functions. DuckDB was also faster for large datasets on disk, whereas pandas is convenient for smaller in-memory data. The differences mattered most for queries like top N per group, running totals, and conditional percentages.

In [228]:
arrow_table = pa.Table.from_pandas(df)
arrow_table

pyarrow.Table
user_id: int64
city: large_string
score: double
active: int64
signup_date: timestamp[ms]
age: int64
sessions: int64
revenue: double
score_rank: double
----
user_id: [[0,1,2,3,4,...,499995,499996,499997,499998,499999]]
city: [["East Timothy","West Erinborough","Caseyport","East Timothy","Smithside",...,"West Erinborough","South Michelletown","Port Jameston","Port Jameston","South Michelletown"],["Port Curtismouth","Carolynland","South Michelletown","New Brandon","Port Curtismouth",...,"Port Stephanieshire","West Erinborough","Port Jameston","West Erinborough","Port Curtismouth"],["Carolynland","Port Stephanieshire","East Timothy","Port Curtismouth","Port Stephanieshire",...,"West Erinborough","East Timothy","Carolynland","New Brandon","South Michelletown"],["West Erinborough","Port Jameston","Carolynland","West Erinborough","South Michelletown",...,"Smithside","New Brandon","Carolynland","New Brandon","West Erinborough"]]
score: [[53.36670363491418,51.81765287440082,82.465

In [229]:
print("Arrow schema:")
print(arrow_table.schema)
print("\nPandas dtypes:")
print(df.dtypes)

Arrow schema:
user_id: int64
city: large_string
score: double
active: int64
signup_date: timestamp[ms]
age: int64
sessions: int64
revenue: double
score_rank: double
-- schema metadata --
pandas: '{"index_columns": [{"kind": "range", "name": null, "start": 0, "' + 1297

Pandas dtypes:
user_id                 int64
city                      str
score                 float64
active                  int64
signup_date    datetime64[ms]
age                     int64
sessions                int64
revenue               float64
score_rank            float64
dtype: object


In [230]:
pq.write_table(arrow_table, "arrow_dataset.parquet")
new_arrow_table = pq.read_table("arrow_dataset.parquet")
new_arrow_table

pyarrow.Table
user_id: int64
city: large_string
score: double
active: int64
signup_date: timestamp[ms]
age: int64
sessions: int64
revenue: double
score_rank: double
----
user_id: [[0,1,2,3,4,...,131067,131068,131069,131070,131071],[131072,131073,131074,131075,131076,...,262139,262140,262141,262142,262143],[262144,262145,262146,262147,262148,...,393211,393212,393213,393214,393215],[393216,393217,393218,393219,393220,...,499995,499996,499997,499998,499999]]
city: [["East Timothy","West Erinborough","Caseyport","East Timothy","Smithside",...,"West Erinborough","South Michelletown","Port Jameston","Port Jameston","South Michelletown"],["Port Curtismouth","Carolynland","South Michelletown","New Brandon","Port Curtismouth",...,"Port Stephanieshire","West Erinborough","Port Jameston","West Erinborough","Port Curtismouth"],["Carolynland","Port Stephanieshire","East Timothy","Port Curtismouth","Port Stephanieshire",...,"West Erinborough","East Timothy","Carolynland","New Brandon","South Michell

In [231]:
df_from_arrow = new_arrow_table.to_pandas()
print(df_from_arrow)

print("Data matches original:", df.equals(df_from_arrow))

        user_id              city      score  active signup_date  age  \
0             0      East Timothy  53.366704       0  2025-05-19   22   
1             1  West Erinborough  51.817653       0  2023-02-27   42   
2             2         Caseyport  82.465913       1  2023-04-27   28   
3             3      East Timothy  36.006728       1  2023-12-20   20   
4             4         Smithside  67.912248       1  2025-07-18   39   
...         ...               ...        ...     ...         ...  ...   
499995   499995         Smithside   9.393500       0  2026-02-07   44   
499996   499996       New Brandon  53.792138       1  2024-12-27   67   
499997   499997       Carolynland  37.259738       1  2025-03-26   26   
499998   499998       New Brandon  74.783441       1  2025-08-30   72   
499999   499999  West Erinborough  36.902064       1  2024-02-27   35   

        sessions     revenue  score_rank  
0             16  230.903998     23256.0  
1            262  229.712312     2405

In [232]:
df_arrow_backed = pd.read_parquet("arrow_dataset.parquet", dtype_backend="pyarrow")
print("Arrow-backed dtypes:")
print(df_arrow_backed.dtypes)

df_traditional = pd.read_parquet("arrow_dataset.parquet")
print("\nTraditional pandas dtypes:")
print(df_traditional.dtypes)

Arrow-backed dtypes:
user_id                int64[pyarrow]
city            large_string[pyarrow]
score                 double[pyarrow]
active                 int64[pyarrow]
signup_date    timestamp[ms][pyarrow]
age                    int64[pyarrow]
sessions               int64[pyarrow]
revenue               double[pyarrow]
score_rank            double[pyarrow]
dtype: object

Traditional pandas dtypes:
user_id                 int64
city                      str
score                 float64
active                  int64
signup_date    datetime64[ms]
age                     int64
sessions                int64
revenue               float64
score_rank            float64
dtype: object


In [233]:
df_arrow_backed = pd.read_parquet("arrow_dataset.parquet", dtype_backend="pyarrow")
print("Arrow-backed dtypes:")
print(df_arrow_backed.dtypes)

df_traditional = pd.read_parquet("arrow_dataset.parquet")
print("\nTraditional pandas dtypes:")
print(df_traditional.dtypes)

Arrow-backed dtypes:
user_id                int64[pyarrow]
city            large_string[pyarrow]
score                 double[pyarrow]
active                 int64[pyarrow]
signup_date    timestamp[ms][pyarrow]
age                    int64[pyarrow]
sessions               int64[pyarrow]
revenue               double[pyarrow]
score_rank            double[pyarrow]
dtype: object

Traditional pandas dtypes:
user_id                 int64
city                      str
score                 float64
active                  int64
signup_date    datetime64[ms]
age                     int64
sessions                int64
revenue               float64
score_rank            float64
dtype: object


Arrow is an in-memory columnar format that connects storage, analysis, and SQL engines. Parquet files on disk can be read into Arrow Tables, which pandas can use efficiently, and DuckDB can query directly. This makes data move from disk to analysis to SQL fast, without extra copying or conversions.